# Chapter 29 Walkthrough: Training Dynamics, Optimizers, Regularization, BatchNorm & Early Stopping

**Book Reference:** Chapter 29, Sections 29.1 – 29.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every deep-learning stabilization experiment uses the bundled `mch_survey_sample` maternal and child health dataset.

In [ ]:
# @title Step 0: Click Play (▶) to Load Packages & MCH Survey Data
!pip install -q sharukh-tutor
import copy
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sharukh_tutor as tutor

torch.set_num_threads(1)
mch_raw = tutor.load_data("mch_survey_sample", as_pandas=True)
mch = mch_raw.copy()
mch["residence"] = (
    mch["residence"].astype(str).str.strip().str.title()
)
mch["anc_visits"] = mch["anc_visits"].replace(999, np.nan)
mch["is_urban"] = (mch["residence"] == "Urban").astype(int)
wealth_map = {
    "Poorest": 1, "Poorer": 2, "Middle": 3,
    "Richer": 4, "Richest": 5,
}
mch["wealth_num"] = mch["wealth_quintile"].map(wealth_map)
feat_cols = [
    "maternal_age_years", "anc_visits", "wealth_num", "is_urban"
]
clean = mch[["maternal_anemia"] + feat_cols].dropna()
X_all = clean[feat_cols].to_numpy(dtype=np.float32)
y_all = clean["maternal_anemia"].astype(int).to_numpy()

X_tr_raw, X_tmp, y_tr, y_tmp = train_test_split(
    X_all, y_all, test_size=0.40, stratify=y_all, random_state=42
)
X_va_raw, X_te_raw, y_va, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42
)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr_raw)
X_va = scaler.transform(X_va_raw)
X_te = scaler.transform(X_te_raw)

Xtr_t = torch.tensor(X_tr, dtype=torch.float32)
ytr_t = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)
Xva_t = torch.tensor(X_va, dtype=torch.float32)
yva_t = torch.tensor(y_va, dtype=torch.float32).unsqueeze(1)
Xte_t = torch.tensor(X_te, dtype=torch.float32)
print(
    f"Split sizes -> train={len(y_tr)}, val={len(y_va)}, "
    f"test={len(y_te)} | Anemia={y_all.mean():.4f}"
)


## Section 29.3.1 — Vanishing/Exploding Variance & He vs. Xavier Initialization

In [ ]:
# Forward activation variance across 10 ReLU layers (width=128)
for label, mode in [
    ("Too small (std=0.02)", 0.02),
    ("Kaiming ReLU", "kaiming"),
    ("Xavier Uniform", "xavier"),
    ("Too large (std=0.35)", 0.35),
]:
    torch.manual_seed(42)
    h = torch.randn(64, 128)
    for _ in range(10):
        layer = nn.Linear(128, 128, bias=False)
        if mode == "kaiming":
            nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
        elif mode == "xavier":
            nn.init.xavier_uniform_(layer.weight)
        else:
            nn.init.normal_(layer.weight, mean=0.0, std=mode)
        h = torch.relu(layer(h))
    print(f"{label:21s} | 10-layer act std = {h.std().item():.6f}")


## Section 29.3.2 — Controlled Optimizer Benchmark: SGD vs. Momentum vs. AdamW

In [ ]:
torch.manual_seed(42)
base_mlp = nn.Sequential(
    nn.Linear(4, 32), nn.ReLU(), nn.Linear(32, 1)
)
base_sd = copy.deepcopy(base_mlp.state_dict())
crit = nn.BCEWithLogitsLoss()

configs = [
    ("SGD", lambda p: torch.optim.SGD(p, lr=0.05)),
    ("SGD+Momentum", lambda p: torch.optim.SGD(p, lr=0.05, momentum=0.9)),
    ("AdamW", lambda p: torch.optim.AdamW(p, lr=0.01, weight_decay=1e-2)),
]
for name, build_opt in configs:
    m = nn.Sequential(nn.Linear(4, 32), nn.ReLU(), nn.Linear(32, 1))
    m.load_state_dict(copy.deepcopy(base_sd))
    opt = build_opt(m.parameters())
    for _ in range(40):
        m.train()
        opt.zero_grad()
        loss = crit(m(Xtr_t), ytr_t)
        loss.backward()
        opt.step()
    m.eval()
    with torch.no_grad():
        tr_l = float(crit(m(Xtr_t), ytr_t).item())
        va_l = float(crit(m(Xva_t), yva_t).item())
        va_p = torch.sigmoid(m(Xva_t)).squeeze(1).numpy()
        va_auc = roc_auc_score(y_va, va_p)
    print(
        f"{name:13s} | train={tr_l:.4f} | "
        f"val={va_l:.4f} | val_auc={va_auc:.4f}"
    )


## Section 29.3.3 — BatchNorm1d & The Dropout `train()` vs. `eval()` Trap

In [ ]:
class ClinicalRegNet(nn.Module):
    def __init__(self, in_dim, hidden_dim=32, dropout_p=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)

torch.manual_seed(42)
reg_net = ClinicalRegNet(in_dim=4, hidden_dim=32, dropout_p=0.3)
with torch.no_grad():
    reg_net.train()
    t1, t2 = reg_net(Xva_t[:8]), reg_net(Xva_t[:8])
    reg_net.eval()
    e1, e2 = reg_net(Xva_t[:8]), reg_net(Xva_t[:8])
print(f"train() max |t1 - t2| = {(t1 - t2).abs().max().item():.5f}")
print(f"eval()  max |e1 - e2| = {(e1 - e2).abs().max().item():.5f}")


## Section 29.3.4 — Gradient Norm Clipping & Cosine Annealing LR Schedule

In [ ]:
# 1. Gradient clipping on an artificially amplified loss
torch.manual_seed(42)
clip_net = ClinicalRegNet(in_dim=4, hidden_dim=32, dropout_p=0.3)
clip_opt = torch.optim.AdamW(clip_net.parameters(), lr=0.01)
clip_opt.zero_grad()
big_loss = crit(clip_net(Xtr_t), ytr_t) * 25.0
big_loss.backward()
norm_before = float(
    nn.utils.clip_grad_norm_(clip_net.parameters(), max_norm=1.0)
)
norm_after = math.sqrt(sum(
    float(p.grad.detach().pow(2).sum().item())
    for p in clip_net.parameters() if p.grad is not None
))
clip_opt.step()
print(
    f"Grad L2 norm -> before={norm_before:.4f} | "
    f"after={norm_after:.4f}"
)

# 2. Smooth CosineAnnealingLR decay across 20 epochs
dummy = nn.Parameter(torch.zeros(1))
lr_opt = torch.optim.SGD([dummy], lr=0.02)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(lr_opt, T_max=20)
lrs = []
for _ in range(20):
    lrs.append(round(float(lr_opt.param_groups[0]["lr"]), 6))
    lr_opt.step()
    sched.step()
print(
    f"Cosine LR -> ep0={lrs[0]:.6f}, "
    f"ep10={lrs[10]:.6f}, ep19={lrs[-1]:.6f}"
)


## Section 29.3.5 — Early Stopping & Restoring the Best Validation Checkpoint

In [ ]:
torch.manual_seed(42)
es_net = ClinicalRegNet(in_dim=4, hidden_dim=32, dropout_p=0.3)
es_opt = torch.optim.AdamW(es_net.parameters(), lr=0.02, weight_decay=1e-2)
es_sched = torch.optim.lr_scheduler.CosineAnnealingLR(es_opt, T_max=80)

best_loss, best_ep, wait, best_sd = float("inf"), 0, 0, None
for ep in range(1, 81):
    es_net.train()
    es_opt.zero_grad()
    crit(es_net(Xtr_t), ytr_t).backward()
    nn.utils.clip_grad_norm_(es_net.parameters(), max_norm=1.0)
    es_opt.step()
    es_sched.step()

    es_net.eval()
    with torch.no_grad():
        vloss = float(crit(es_net(Xva_t), yva_t).item())
    if vloss < best_loss:
        best_loss, best_ep, wait = vloss, ep, 0
        best_sd = copy.deepcopy(es_net.state_dict())
    else:
        wait += 1
        if wait >= 8:
            break

es_net.load_state_dict(best_sd)
es_net.eval()
with torch.no_grad():
    te_probs = torch.sigmoid(es_net(Xte_t)).squeeze(1).numpy()
te_auc = roc_auc_score(y_te, te_probs)
print(
    f"Stopped at ep={ep} (best_ep={best_ep}) | "
    f"best_val={best_loss:.5f} | test_auc={te_auc:.4f}"
)


## Section 29.3.6 — Break It On Purpose! (Training Dynamics Error Anatomy)

In [ ]:
# Intentional Bug #1: BatchNorm1d crashes on batch size 1 in train() mode
bn = nn.BatchNorm1d(8)
bn.train()
bn(torch.randn(1, 8))


In [ ]:
# Intentional Bug #2: Negative learning rate passed to AdamW
torch.optim.AdamW([nn.Parameter(torch.zeros(2))], lr=-0.01)


In [ ]:
# Intentional Bug #3: Dropout probability outside [0, 1]
nn.Dropout(p=1.5)


## Section 29.4 — Stage 4 Blueprint: Stabilized Clinical Training Harness

In [ ]:
def train_stabilized_clinical_net(
    raw_df, target="maternal_anemia", seed=42
):
    """Train Kaiming+BN+Dropout MLP with AdamW, cosine LR & early stop."""
    df = raw_df.copy()
    df["residence"] = df["residence"].astype(str).str.strip().str.title()
    df["anc_visits"] = df["anc_visits"].replace(999, np.nan)
    df["is_urban"] = (df["residence"] == "Urban").astype(int)
    w_map = {
        "Poorest": 1, "Poorer": 2, "Middle": 3,
        "Richer": 4, "Richest": 5,
    }
    df["wealth_num"] = df["wealth_quintile"].map(w_map)
    cols = ["maternal_age_years", "anc_visits", "wealth_num", "is_urban"]
    sub = df[[target] + cols].dropna()
    X = sub[cols].to_numpy(dtype=np.float32)
    y = sub[target].astype(int).to_numpy()
    Xtr, Xtmp, ytr, ytmp = train_test_split(
        X, y, test_size=0.40, stratify=y, random_state=seed
    )
    Xva, Xte, yva, yte = train_test_split(
        Xtmp, ytmp, test_size=0.50, stratify=ytmp, random_state=seed
    )
    sc = StandardScaler()
    Xtr = sc.fit_transform(Xtr)
    Xva = sc.transform(Xva)
    Xte = sc.transform(Xte)
    torch.manual_seed(seed)
    net = ClinicalRegNet(in_dim=4, hidden_dim=32, dropout_p=0.3)
    for m in net.modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            nn.init.zeros_(m.bias)
    opt = torch.optim.AdamW(net.parameters(), lr=0.02, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=80)
    loss_fn = nn.BCEWithLogitsLoss()
    Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
    ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1)
    Xva_t = torch.tensor(Xva, dtype=torch.float32)
    yva_t = torch.tensor(yva, dtype=torch.float32).unsqueeze(1)
    Xte_t = torch.tensor(Xte, dtype=torch.float32)
    best_l, wait, best_sd = float("inf"), 0, None
    for ep in range(1, 81):
        net.train()
        opt.zero_grad()
        loss_fn(net(Xtr_t), ytr_t).backward()
        nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        opt.step()
        sched.step()
        net.eval()
        with torch.no_grad():
            vl = float(loss_fn(net(Xva_t), yva_t).item())
        if vl < best_l:
            best_l, wait = vl, 0
            best_sd = copy.deepcopy(net.state_dict())
        else:
            wait += 1
            if wait >= 8:
                break
    net.load_state_dict(best_sd)
    net.eval()
    with torch.no_grad():
        p_te = torch.sigmoid(net(Xte_t)).squeeze(1).numpy()
    return {
        "epochs_run": ep,
        "best_val_loss": round(best_l, 5),
        "test_auc": round(float(roc_auc_score(yte, p_te)), 4),
    }

for label, col in [
    ("Anemia", "maternal_anemia"),
    ("Stunting", "child_stunted"),
]:
    res = train_stabilized_clinical_net(mch_raw, target=col)
    print(
        f"{label:8s} | epochs={res['epochs_run']:2d} | "
        f"best_val={res['best_val_loss']:.5f} | "
        f"test_auc={res['test_auc']:.4f}"
    )
